<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v2/MNPS_Job_Classification_Kimi_HuggingFace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# MNPS Job Classification - Kimi via HuggingFace Router FIXED

**Version 8.0 - HuggingFace Router Edition FIXED**

## Changes in this version:
- ✅ Uses HuggingFace Router instead of direct Moonshot API
- ✅ Improved JSON parsing with 3-tier validation
- ✅ Better error handling and retry logic
- ✅ Uses Kimi-K2-Instruct model via Groq
- ✅ Enhanced brace-matching for complete JSON extraction

## Important Notes:
- **API Required:** HuggingFace Token (get at https://huggingface.co/settings/tokens)
- **Cost:** Pay-per-use through HuggingFace (typically lower than direct API)
- **Speed:** 4-10 seconds per job (two-pass)
- **Model:** moonshotai/Kimi-K2-Instruct-0905:groq

# **Section 1: Setup, Imports, Mount Drive**

In [ ]:
# ==== CELL 1: Imports and Setup ====
import os, json, zipfile, re, time, random
from pathlib import Path
from datetime import datetime
from typing import Dict, Tuple, Optional

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from google.colab import drive, userdata
from openai import OpenAI

# Mount Drive
print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# Create run folder
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = Path('/content')
OUTPUTS_DIR = Path(f"/content/drive/My Drive/Colab Notebooks/Run Results/RUN_{timestamp}_Kimi/outputs")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Outputs directory: {OUTPUTS_DIR}")
print("✅ Setup complete")

# **Section 2: Kimi API Configuration**

In [ ]:
# ==== CELL 2: Kimi via HuggingFace Router Configuration ====
# Get HuggingFace token from Colab secrets panel
# To add: Click the key icon in left sidebar > Add new secret > Name: HF_TOKEN
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN is empty")
except Exception as e:
    print("❌ Error: HF_TOKEN not found in Colab secrets!")
    print("   Please add your HuggingFace token:")
    print("   1. Click the key icon (🔑) in the left sidebar")
    print("   2. Click 'Add new secret'")
    print("   3. Name: HF_TOKEN")
    print("   4. Value: Your HuggingFace token from https://huggingface.co/settings/tokens")
    raise

# Initialize client with HuggingFace Router
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=HF_TOKEN
)

# Model selection - Kimi K2 via Groq on HuggingFace
MODEL_ID = "moonshotai/Kimi-K2-Instruct-0905:groq"

print(f"✅ Kimi client initialized via HuggingFace Router")
print(f"   Model: {MODEL_ID}")
print(f"   Base URL: https://router.huggingface.co/v1")
print(f"   This uses HuggingFace's routing for better availability")

# **Section 3: Data Loading**

In [ ]:
# ==== CELL 3: Robust Data Loading with Zip Extraction ====
# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - please upload it to /content/")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# If main input file not found, check inside the zip
if not BATCH_INPUT_CSV.exists() and ZIP_FILE.exists():
    print("⚠️  Sample JDs.csv not found in root — checking inside MNPS Prompt Resources.zip...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()
        if "Sample JDs.csv" in zip_contents:
            zip_ref.extract("Sample JDs.csv", RUN_ROOT)
            print("✅ Extracted Sample JDs.csv from zip")
        else:
            print("❌ Sample JDs.csv not found in zip contents:", zip_contents)
            raise FileNotFoundError("Sample JDs.csv not found in root or zip")

    # Optionally extract Ground Truth if present
    if "Ground Truth Masterfile.csv" in zip_contents:
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extract("Ground Truth Masterfile.csv", RUN_ROOT)
            print("✅ Extracted Ground Truth Masterfile.csv from zip")

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

# Verify all files exist
required_files = {
    "Sample JDs": BATCH_INPUT_CSV,
    "MNPS Roles": MNPS_ROLES_CSV,
    "MNPS KSACs": MNPS_KSACS_CSV,
    "Competencies": COMPETENCY_EXTENDED_CSV,
    "Korn Ferry": KORN_FERRY_CSV
}

print(f"\n{'='*60}")
print("📄 FILE VERIFICATION")
print("="*60)
for name, path in required_files.items():
    if not path.exists():
        print(f"❌ MISSING: {name} at {path}")
    else:
        print(f"✅ Found: {path.name}")

# Load all datasets
try:
    df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
    gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1') if GT_MASTERFILE_CSV.exists() else None
    roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
    ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
    competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
    korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

    print(f"\n{'='*60}")
    print("📊 DATASET SUMMARY")
    print("="*60)
    print(f"✅ Job descriptions: {len(df)}")
    if gt_df is not None:
        print(f"✅ Ground truth records: {len(gt_df)}")
    print(f"✅ MNPS roles: {len(roles_df)}")
    print(f"✅ MNPS KSACs: {len(ksacs_df)}")
    print(f"✅ Competency descriptions: {len(competency_df)}")
    print(f"✅ Korn Ferry competencies: {len(korn_ferry_df)}")
    print(f"{'='*60}\n")
except Exception as e:
    print(f"❌ Error loading data: {e}")
    raise

# **Section 4: Build KSACs Context**

In [ ]:
# ==== CELL 4: Build Comprehensive KSACs Text ====
def build_ksacs_text() -> str:
    """Construct full KSACs context for Kimi."""
    ksacs_text = "MNPS KNOWLEDGE, SKILLS, ABILITIES, AND COMPETENCIES:\n\n"

    # Role-specific KSACs
    role_col = next((c for c in ksacs_df.columns if 'role' in c.lower()), None)
    ksacs_col = next((c for c in ksacs_df.columns if 'ksacs' in c.lower() or 'ksac' in c.lower()), None)
    if role_col and ksacs_col:
        for _, row in ksacs_df.iterrows():
            if pd.notna(row.get(role_col)) and pd.notna(row.get(ksacs_col)):
                ksacs_text += f"**{row[role_col]}**:\n{row[ksacs_col]}\n\n"

    # Competency descriptions
    comp_col = next((c for c in competency_df.columns if 'competency' in c.lower()), None)
    desc_col = next((c for c in competency_df.columns if 'description' in c.lower()), None)
    if comp_col and desc_col:
        ksacs_text += "**COMPETENCY EXTENDED DESCRIPTIONS**:\n"
        for _, row in competency_df.iterrows():
            if pd.notna(row.get(comp_col)) and pd.notna(row.get(desc_col)):
                ksacs_text += f"- {row[comp_col]}: {row[desc_col]}\n"

    # Korn Ferry competencies
    kf_comp = next((c for c in korn_ferry_df.columns if 'competency' in c.lower()), None)
    kf_desc = next((c for c in korn_ferry_df.columns if 'definition' in c.lower() or 'description' in c.lower()), None)
    if kf_comp and kf_desc:
        ksacs_text += "\n**KORN FERRY 38 COMPETENCIES**:\n"
        for _, row in korn_ferry_df.iterrows():
            if pd.notna(row.get(kf_comp)) and pd.notna(row.get(kf_desc)):
                ksacs_text += f"- {row[kf_comp]}: {row[kf_desc]}\n"

    return ksacs_text

KSACS_TEXT = build_ksacs_text()
print(f"✅ Built KSACs text ({len(KSACS_TEXT):,} characters)")
print(f"   Will use first 15,000 chars in prompts to stay within token limits")

# **Section 5: Classification Rules & Constants**

In [ ]:
# ==== CELL 5: Enhanced Classification Rules ====
VALID_ROLES = roles_df.iloc[:, 0].dropna().tolist()

# Roles that should NOT have minor sub-grouping
EXEMPT_FROM_MINOR = {
    'Teacher', 'Principal', 'Assistant Principal', 'Counselor',
    'Librarian', 'Therapist', 'Social Worker', 'Instructor'
}

# Executive roles (rarely "Lead")
EXECUTIVE_ROLES = {'Coordinator', 'Principal', 'Director', 'Manager'}

# Minor role mapping
CANON_MINOR_MAP = {
    'i': 'I', '1': 'I', 'one': 'I', 'entry': 'I',
    'ii': 'II', '2': 'II', 'two': 'II',
    'iii': 'III', '3': 'III', 'three': 'III',
    'lead': 'Lead', 'iv': 'III', '4': 'III'
}

# Specialist fallback patterns
SPECIALIST_PATTERNS = [
    ('Technician', r'technical|repair|maintenance|install|troubleshoot|equipment|hands-on|tools'),
    ('Analyst', r'analyz|data research|evaluat|statistical|metrics|reports'),
    ('Teacher', r'classroom|curriculum|instruction|students'),
    ('Coach', r'instructional coach|mentor|professional development|co-teach'),
    ('Accountant', r'accounting|financial|audit|budget|fiscal'),
    ('Coordinator', r'coordinate|organize|facilitate|liaison'),
]

print("✅ Classification rules loaded")
print(f"   Valid roles: {len(VALID_ROLES)}")
print(f"   Exempt from minor: {len(EXEMPT_FROM_MINOR)}")
print(f"   Specialist patterns: {len(SPECIALIST_PATTERNS)}")

# **Section 6: Post-Processing Functions**

In [ ]:
# ==== CELL 6: Enhanced JSON Parsing ====

def extract_json_from_text(text: str) -> Optional[str]:
    """Extract JSON from text with 3-tier validation."""

    # ✅ FIXED: Ensure text is a string
    if not isinstance(text, str):
        print(f"❌ Input is not a string: {type(text)}")
        return None

    # Tier 1: Direct parse (if already clean JSON)
    try:
        json.loads(text)
        return text
    except:
        pass

    # Tier 2: Remove markdown/backticks
    cleaned = text.strip()
    # ✅ FIXED: Correct regex patterns (was \\s, now \s)
    cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    cleaned = cleaned.strip()

    try:
        json.loads(cleaned)
        return cleaned
    except:
        pass

    # Tier 3: Brace matching extraction
    json_match = None
    stack = []
    start_idx = None

    for i, char in enumerate(text):
        if char == '{':
            if not stack:
                start_idx = i
            stack.append(char)
        elif char == '}' and stack:
            stack.pop()
            if not stack and start_idx is not None:
                json_match = text[start_idx:i+1]
                break

    if json_match:
        try:
            json.loads(json_match)
            return json_match
        except:
            pass

    return None

def parse_kimi_response(response_text: str) -> Dict[str, str]:
    """Parse Kimi's response with fallback handling."""

    default_response = {
        "new_job_title": "Unknown",
        "major_role_group": "Other",
        "minor_sub_group": "",
        "grouping_justification": "Could not parse response"
    }

    # ✅ Debug log to see what we received
    print(f"📥 Raw response: {str(response_text)[:200]}...")

    if not response_text:
        print("❌ Empty response received")
        return default_response

    # Try to extract JSON
    json_str = extract_json_from_text(response_text)
    if not json_str:
        print("❌ Could not extract valid JSON structure")
        return default_response

    try:
        parsed = json.loads(json_str)

        # ✅ Ensure all required fields exist
        result = {
            "new_job_title": parsed.get("new_job_title", "Unknown"),
            "major_role_group": parsed.get("major_role_group", "Other"),
            "minor_sub_group": parsed.get("minor_sub_group", ""),
            "grouping_justification": parsed.get("grouping_justification", "")
        }

        print(f"✅ Parsed successfully: {result['major_role_group']}")
        return result

    except Exception as e:
        print(f"❌ JSON parsing failed: {e}")
        return default_response

print("✅ JSON parsing utilities loaded")

# **Section 7: Kimi API Helper Functions**

In [ ]:
# ==== CELL 7: Kimi API Function with Retry Logic ====

def call_kimi_api(prompt: str, max_retries: int = 3) -> Optional[str]:
    """Call Kimi API with retry logic and type safety."""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                max_tokens=300,
                top_p=0.1
            )

            # ✅ FIXED: Ensure content is a string
            content = response.choices[0].message.content

            if not isinstance(content, str):
                print(f"⚠️ Warning: API returned non-string content: {type(content)}")
                return None

            return content

        except Exception as e:
            error_msg = str(e)

            if "rate_limit" in error_msg.lower():
                wait_time = min(2 ** attempt * 5, 30)
                print(f"  Rate limit hit, waiting {wait_time}s...")
                time.sleep(wait_time)
                continue

            elif attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"  API error on attempt {attempt + 1}: {error_msg}")
                print(f"  Retrying in {wait_time}s...")
                time.sleep(wait_time)
                continue

            else:
                print(f"  API failed after {max_retries} attempts: {error_msg}")
                return None

    return None

print("✅ Kimi API function loaded")

# **Section 8: Prompt Templates**

In [ ]:
# ==== CELL 8: Kimi-Optimized Prompts (FIXED) ====
PROBLEM_ROLE_CLARIFICATIONS = """
**CRITICAL CLASSIFICATION RULES:**

1. **ROLE DISTINCTION - Coordinator vs Manager:**
   - **Coordinator**: Facilitates, organizes, liaises, manages PROGRAMS (NO hire/fire authority)
   - **Manager**: Has AUTHORITY to hire/fire staff, control budgets, develop POLICIES
   - DECISION: If justification mentions "coordinate" ≥3 times but NOT "supervise staff" → Coordinator

2. **EXEMPT ROLES - No Minor Sub-Grouping:**
   Teacher, Principal, Assistant Principal, Counselor, Librarian, Therapist, Social Worker, Instructor
   → minor_sub_group MUST be empty string ""

3. **AUTO-ELEVATION TRIGGERS:**
   - Doctoral degree requirement → "III"
   - "Lead" in original title + supervises staff → "Lead"
   - Master's + 5+ years leadership → "III"

4. **SELF-CHECK BEFORE OUTPUT:**
   - Does my major_role_group appear VERBATIM in my justification?
   - If not, reclassify based on justification's primary verb
"""

# Base classification prompt
ZERO_SHOT_PROMPT = f"""
Objective: Classify MNPS jobs by function attributes only, ignoring titles.

Process:
- Analyze: Education, Experience, Essential Functions, KSACs
- Compare against MNPS standards and competency frameworks
- Provide detailed justification aligned to selected role

{PROBLEM_ROLE_CLARIFICATIONS}

**MNPS Resources:**
{{ksacs_text}}

**Available Roles:** {{valid_roles}}

**Job to Classify:**
{{job_text}}

**OUTPUT AS JSON with these exact keys:**
{{
  "major_role_group": "<one of the available roles>",
  "minor_sub_group": "<I, II, III, Lead, or empty string for exempt roles>",
  "new_job_title": "<suggested standardized title>",
  "grouping_justification": "<detailed explanation that MUST mention the major_role_group>"
}}
"""

# Self-consistency check prompt
SELF_CONSISTENCY_PROMPT = """
Review your previous classification for internal consistency.

**Your Previous Output:**
{pass1_output}

**Self-Check Questions:**
1. Does the `grouping_justification` explicitly mention the `major_role_group` you selected?
2. Based on the primary verbs in the justification, is the classification accurate?
3. Is the `minor_sub_group` appropriate for this `major_role_group`?

**Task:**
If there are inconsistencies, provide a CORRECTED classification.
If everything is consistent, return the SAME classification.

**OUTPUT AS JSON with these exact keys:**
{{
  "major_role_group": "<corrected or same>",
  "minor_sub_group": "<corrected or same>",
  "new_job_title": "<corrected or same>",
  "grouping_justification": "<original or improved justification>",
  "changes_made": "<description of any corrections, or 'none'>"
}}
"""

print("✅ Prompt templates loaded")
print(f"   ZERO_SHOT_PROMPT: {len(ZERO_SHOT_PROMPT):,} characters")
print(f"   SELF_CONSISTENCY_PROMPT: {len(SELF_CONSISTENCY_PROMPT):,} characters")

# **Section 9: Main Processing Function**

In [ ]:
# ==== CELL 9: Main Processing Function (FIXED) ====
def process_job_with_kimi(row_idx: int, row: pd.Series) -> dict:
    """Two-pass Kimi classification with enhanced post-processing."""

    job_text = get_job_text(row)

    try:
        # === PASS 1: Initial Classification ===
        base_prompt = ZERO_SHOT_PROMPT.format(
            ksacs_text=KSACS_TEXT[:15000],  # Limit to stay within token budget
            valid_roles=', '.join(VALID_ROLES[:50]),  # Limit roles list
            job_text=job_text[:3000]  # Limit job text
        )

        pass1_result = call_kimi_json_with_retry(base_prompt)

        # Extract values
        major = pass1_result.get('major_role_group', 'Other')
        minor = pass1_result.get('minor_sub_group', 'I')
        justification = pass1_result.get('grouping_justification', '')

        # === PYTHON PRE-CORRECTIONS (before Pass 2) ===
        # Apply specialist fallback
        if major == 'Specialist':
            for role_name, pattern in SPECIALIST_PATTERNS:
                if re.search(pattern, job_text, re.I):
                    major = role_name
                    break

        # === PASS 2: Self-Consistency Check ===
        pass2_prompt = SELF_CONSISTENCY_PROMPT.format(
            pass1_output=json.dumps(pass1_result, indent=2)
        )

        pass2_result = call_kimi_json_with_retry(pass2_prompt)

        # === ENHANCED POST-PROCESSING ===
        final_major = pass2_result.get('major_role_group', major)
        final_minor = pass2_result.get('minor_sub_group', minor)
        final_justification = pass2_result.get('grouping_justification', justification)

        # 1. Enforce justification alignment
        final_major = enforce_role_justification_alignment(
            job_text, final_major, final_justification
        )

        # 2. Auto-elevate by credentials
        final_major, final_minor = auto_elevate_by_credentials(
            row, final_major, final_minor
        )

        # 3. Apply minor exemptions
        final_minor = apply_minor_exemption(final_major, final_minor)

        # 4. Fix executive minor grouping
        final_minor = fix_executive_minor(final_major, final_minor)

        # 5. Normalize
        final_minor = normalize_minor(final_minor)

        # Build final title
        final_title = f"{final_major} {final_minor}".strip() if final_minor else final_major

        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': final_title,
            'major_role_group': final_major,
            'minor_sub_group': final_minor,
            'grouping_justification': final_justification,
            'model_used': MODEL_ID,
            'pass1_raw': major,
            'corrections_applied': final_major != major or final_minor != minor
        }

    except Exception as e:
        print(f"❌ Error on row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': 'ERROR',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'ERROR: {str(e)}',
            'model_used': MODEL_ID,
            'pass1_raw': 'ERROR',
            'corrections_applied': False
        }

print("✅ Processing function loaded")

# **Section 10: Execute Batch Processing**

In [ ]:
# ==== CELL 10: Run Classifications ====
results = []
corrections = []

print(f"🚀 Processing {len(df)} jobs with Kimi...")
print(f"   Model: {MODEL_ID}")
print(f"   Two-pass with self-consistency check\n")

# Track timing
start_time = time.time()

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying"):
    # ✅ FIXED: Wrap in try-except for graceful error handling
    try:
        result = process_job_with_kimi(idx, row)

        # ✅ FIXED: Validate result is a dictionary
        if not isinstance(result, dict):
            print(f"❌ Row {idx}: Invalid result type {type(result)}, creating error entry")
            result = {
                'source_row_index': idx,
                'job_title_original': row.get('Job Title', 'Unknown'),
                'new_job_title': 'Error',
                'major_role_group': 'ERROR',
                'minor_sub_group': '',
                'grouping_justification': 'Invalid result type',
                'corrections_applied': False,
                'model_used': MODEL_ID
            }

        results.append(result)

        # ✅ FIXED: Safely access dictionary keys with .get()
        if result.get('corrections_applied', False):
            corrections.append({
                'row': idx,
                'title': result.get('job_title_original', 'Unknown'),
                'corrected_to': result.get('major_role_group', 'ERROR'),
                'reason': 'Post-processing correction'
            })

    except Exception as e:
        print(f"❌ Unexpected error on row {idx}: {e}")
        results.append({
            'source_row_index': idx,
            'job_title_original': row.get('Job Title', 'Unknown'),
            'new_job_title': 'Error',
            'major_role_group': 'EXCEPTION',
            'minor_sub_group': '',
            'grouping_justification': f'Exception: {str(e)}',
            'corrections_applied': False,
            'model_used': MODEL_ID
        })

    # Rate limit protection
    time.sleep(0.5)

# Calculate timing
elapsed_time = time.time() - start_time
avg_time = elapsed_time / len(df)

print(f"\n✅ Processing complete!")
print(f"   Total time: {elapsed_time:.1f} seconds")
print(f"   Average per job: {avg_time:.1f} seconds")
print(f"   Corrections applied: {len(corrections)}")

# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Kimi_v1.csv"
results_df.to_csv(output_path, index=False)

# Save corrections log
if corrections:
    corrections_df = pd.DataFrame(corrections)
    corrections_path = OUTPUTS_DIR / "corrections_log.csv"
    corrections_df.to_csv(corrections_path, index=False)
    print(f"\n✅ Saved corrections log: {corrections_path}")

print(f"\n✅ Saved results to: {output_path}")

# **Section 11: Validation & Summary**

In [ ]:
# ==== CELL 11: Generate Summary Statistics ====
def generate_summary(preds_df: pd.DataFrame):
    """Create summary report of classifications."""

    stats = {
        'total_jobs': len(preds_df),
        'unique_roles': preds_df['major_role_group'].nunique(),
        'minor_distribution': preds_df['minor_sub_group'].value_counts().to_dict(),
        'corrections_applied': int(preds_df['corrections_applied'].sum()),
        'errors': int((preds_df['major_role_group'] == 'Other').sum()),
        'specialist_count': int((preds_df['major_role_group'] == 'Specialist').sum()),
        'exempt_with_minor': int(sum(
            (preds_df['major_role_group'].isin(EXEMPT_FROM_MINOR)) &
            (preds_df['minor_sub_group'] != '')
        ))
    }

    # Save detailed stats
    stats_path = OUTPUTS_DIR / "summary_statistics.json"
    with open(stats_path, 'w') as f:
        json.dump(stats, f, indent=2)

    # Print summary
    print("\n" + "="*50)
    print("📊 CLASSIFICATION SUMMARY")
    print("="*50)
    print(f"Total Jobs Processed: {stats['total_jobs']}")
    print(f"Unique Major Roles: {stats['unique_roles']}")
    print(f"Corrections Applied: {stats['corrections_applied']}")
    print(f"Errors: {stats['errors']}")
    print(f"\nMinor Role Distribution:")
    for k, v in stats['minor_distribution'].items():
        print(f"  {k}: {v}")

    print(f"\n⚠️  Exempt roles with minor grouping (SHOULD BE 0): {stats['exempt_with_minor']}")
    print(f"⚠️  Specialist count (should be minimal): {stats['specialist_count']}")

    print(f"\n✅ Saved statistics to: {stats_path}")

    return stats

summary = generate_summary(results_df)

# **Section 12: Export for Comparison**

In [ ]:
# ==== CELL 12: Export Results for Comparison ====
# Merge with original data for side-by-side analysis
export_df = df.copy()
export_df = export_df.merge(
    results_df[['source_row_index', 'new_job_title', 'major_role_group',
                'minor_sub_group', 'grouping_justification', 'corrections_applied']],
    left_index=True,
    right_on='source_row_index',
    how='left'
)

# Save detailed results
comparison_path = OUTPUTS_DIR / "detailed_classifications_with_justifications.csv"
export_df.to_csv(comparison_path, index=False)

# Save sample of 10 for quick review
sample_path = OUTPUTS_DIR / "sample_classifications.csv"
export_df.head(10).to_csv(sample_path, index=False)

print(f"✅ Detailed results: {comparison_path}")
print(f"✅ Sample file: {sample_path}")

print("\n" + "="*60)
print("✅ ALL PROCESSING COMPLETE!")
print("="*60)
print(f"\n📁 All results saved to: {OUTPUTS_DIR}")
print(f"\n📄 Main files:")
print(f"   - Job_Classifications_Kimi_v1.csv")
print(f"   - detailed_classifications_with_justifications.csv")
print(f"   - summary_statistics.json")
if corrections:
    print(f"   - corrections_log.csv")
print(f"\n🎉 Job classification completed successfully!")